In [1]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content="Hello world",
    metadata={"source":"https://www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello world')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [6]:
# text data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/Python.txt",encoding="utf-8")

C:\Users\shrey\AppData\Local\Temp\ipykernel_17980\2657093629.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [7]:
document = loader.load()

In [8]:
# document


In [9]:
# PDF data
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

In [10]:
# PDF data
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

### ingestion pipeline

In [11]:
# data -> documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

### Documents

In [12]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            #complete file path
            pdf_path = os.path.join(folder_path,filename)

            loader = PyPDFLoader(pdf_path)
            doc =  loader.load()

            all_docs.extend(doc)
            num_docs +=1
    print("total pdfs: ",num_docs)
    print("total pages : ",len(all_docs))
    return all_docs
    

In [13]:
all_pdf_documents = load_all_pdfs()

total pdfs:  2
total pages :  32


In [14]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

### Chunks

In [15]:
# chunks
!pip install langchain_text_splitter

ERROR: Could not find a version that satisfies the requirement langchain_text_splitter (from versions: none)
ERROR: No matching distribution found for langchain_text_splitter


In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
def split_doc(documents,chunk_size = 500,chunk_overlap = 50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
        
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs
    

In [18]:
chunks = split_doc(all_pdf_documents)

In [19]:
len(chunks)

320

### embedding 

In [20]:
from sentence_transformers import SentenceTransformer

In [21]:
class EmbeddingManager:
    def __init__(self,model_name = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("loading model .....",self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions",self.model.get_sentence_embedding_dimension())
        

    def generate_embeddings(self,text):
        embeddings = self.model.encode(text,show_progress_bar = True)
        print("embeddings shape :",embeddings.shape)
        return embeddings
        


In [22]:
embedding_manager = EmbeddingManager()

loading model ..... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions 384


C:\Users\shrey\AppData\Local\Temp\ipykernel_17980\938003576.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions",self.model.get_sentence_embedding_dimension())


### store vector store database

In [23]:
import chromadb
import uuid

In [24]:
class VectorStoreManger:
    def __init__(self,persist_directory="data/vector_store",collection_name = "pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory,exist_ok = True)
        
        #create a client
        self.client =  chromadb.PersistentClient(path=self.persist_directory)

        #Create the collection
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata = {"description":"vector store collection for pdf embeddings in RAG"}
        )
        print("Initialized the vector store with collection",self.collection_name)
        print("docs int collection",self.collection.count())

    def add_documents(self,documents,embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("number of documents does not match num of embeddings")

        #store - > ids,embedding,document,metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc,embedding) in enumerate (zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4()}" #doc_a1,doc_a2,doc_a3
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = 1
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids =ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store = ",len(documents_content))
        print("docs  in collection ",self.collection.count())
            
    

In [25]:
vector_store = VectorStoreManger()

Initialized the vector store with collection pdf_documents
docs int collection 1280


In [26]:
# data => documents => chunks => embeddings => store in vector store\

texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks,embedding)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

embeddings shape : (320, 384)
total documents added in vector store =  320
docs  in collection  1600


## Retrieval Pipeline 

In [27]:
from sklearn.metrics.pairwise import cosine_similarity

In [28]:
class RAGRetriever:
    def __init__(self,embedding_manager,vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self,query,top_k=5,score_threshold=0.0):
        #query -> embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        #retrieve data from vector store
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )

        #cosine similarity
        retrieved_docs = []
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i ,(doc_id,metadata,document,distance)  in enumerate(zip(ids,metadatas,documents,distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "ids": doc_id,
                        "document":document,
                        "metadata":distance,
                        "similarity_score":similarity_score,
                        "rank": i - 1
                    })
            print(f"retrieved {len(retrieved_docs)} documents")
            

        else:
            print("no document found")

        return retrieved_docs
        
        

In [29]:
rag_retriever = RAGRetriever(embedding_manager,vector_store)

In [30]:
rag_retriever.retrieve("What is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape : (1, 384)
retrieved 5 documents


[{'ids': 'doc_b7df5dcb-37a0-4879-8ff3-3bbb26a6db40',
  'document': 'positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop = 0.1.\n7',
  'metadata': 0.7821967601776123,
  'similarity_score': 0.2178032398223877,
  'rank': -1},
 {'ids': 'doc_2de1abf5-9ea9-4bbb-9787-c3b59ae9566b',
  'document': 'positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop = 0.1.\n7',
  'metadata': 0.7821967601776123,
  'similarity_score': 0.2178032398223877,
  'rank': 0},
 {'ids': 'doc_25cff381-d35d-4802-9af8-1a53a123803e',
  'document': 'positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop = 0.1.\n7',
  'metadata': 0.7821967601776123,
  'similarity_score': 0.2178032398223877,
  'rank': 1},
 {'ids': 'doc_579ad912-c07d-4b19-b3b9-f17a0ffce949',
  'document': 'positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop =

## Integrate with LLMS 

#### OpenAI - GPT

In [31]:
API_KEY_OPENAI = ""

In [32]:
# !pip install langchain-openai

In [41]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key = API_KEY_OPENAI,
    model = "gpt-6.2-astra",
    temperature=1,
    max_tokens=1024
)

In [44]:
# generate our retrieval- augmented output
def generate_output(query,rag_retriever,llm,top_k=3):
    results = rag_retriever.retrieve(query,top_k)

    context ="\n".join([doc["document"] for doc in results])if results else ""

    if not context:
        print("we found not relevant context for the given query")
    
    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context:{context}
                Query:{query} """

    response = llm.invoke(prompt) # expecting a string as prompt
    return response.content
    

In [45]:
answer = generate_output("What is encoder-decoder?",rag_retriever,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape : (1, 384)
retrieved 3 documents


In [36]:
# !pip install --upgrade brotli

In [46]:
print(answer)

An **encoder–decoder** is a model architecture in which the encoder turns an input sequence into a set of representations, and the decoder uses those representations to generate an output sequence one token at a time. In the Transformer described here, each of the six decoder layers uses **encoder–decoder attention** (also called *cross-attention*) to focus on the parts of the encoder’s output most relevant to predicting the next token.


### Groq

In [74]:
API_Key_GROQ = ""

In [75]:
# !pip install langchain-groq

In [85]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key = API_Key_GROQ,
    model = "qwen/qwen3.8-27b",
    temperature=0.1,
    max_tokens=1024
)

In [86]:
# generate our retrieval- augmented output
def generate_output(query,rag_retriever,llm,top_k=3):
    results = rag_retriever.retrieve(query,top_k)

    context ="\n".join([doc["document"] for doc in results])if results else ""

    if not context:
        print("we found not relevant context for the given query")
    
    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context:{context}
                Query:{query} """

    response = llm.invoke([prompt.format(context=context,query=query)]) # expecting a list as prompt
    return response.content

In [87]:
answer = generate_output("What is encoder-decoder?",rag_retriever,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape : (1, 384)
retrieved 3 documents


In [88]:
# from groq import Groq

# # Pass your variable directly instead of using os.environ.get()
# client = Groq(api_key=API_Key_GROQ) 

# # Retrieve and print the active models
# models = client.models.list()
# for model in models.data:
#     if getattr(model, "active", True):
#         print(model.id)

In [89]:
print(answer)

Based on the provided context, the **encoder-decoder** architecture consists of two main components:

1.  **Encoder**: Produces outputs of dimension $d_{model} = 512$.
2.  **Decoder**:
    *   Composed of a stack of $N = 6$ identical layers.
    *   Contains **three sub-layers** in each layer (in addition to the two sub-layers found in the encoder).
    *   The third sub-layer performs **multi-head attention** over the output of the encoder stack.
    *   Uses **residual connections** around each sub-layer, followed by **layer normalization**.
    *   Produces outputs of dimension $d_{model} = 512$.

In summary, the decoder extends the encoder’s structure by adding a cross-attention mechanism (multi-head attention over encoder outputs) while maintaining the same output dimensionality and using residual connections and layer normalization.
